### Установка библиотек

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

from datasets import load_dataset
import random
import numpy as np
import pandas as pd
import json

from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_cosine_schedule_with_warmup

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

### Загрузка датасета для детекции токсичности

In [2]:
path = kagglehub.dataset_download("blackmoon/russian-language-toxic-comments")
path = os.path.join(path, "labeled.csv")

### Деление на тренировочную, валидационную и тестовую выборки

In [ ]:
data = pd.read_csv(path)
ds_train, ds_test = train_test_split(data, test_size=0.1)
ds_val, ds_test = train_test_split(ds_test, test_size=0.5)

### Токенизатор

In [20]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

### Реализация срезов

In [ ]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                [sentences[i]],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        item = {key: val.squeeze(0) for key, val in encoding.items()} 
        item['labels'] = torch.tensor(labels[i], dtype=torch.long) 
        result.append(item)
    return result

### Кастомный датасет

In [ ]:
class ToxicDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences.tolist()
        self.labels = labels.astype(float).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]
            
            encoding = self.tokenizer(
                [tokens],
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            item = {key: val.squeeze(0) for key, val in encoding.items()} 
            item['labels'] = torch.tensor(tag, dtype=torch.long)
            return item
           

### Параметры

In [21]:
batch_size = 16
max_length = 512
epochs = 3

### Создание датасетов для тренировки, валидации и тестирования

In [22]:
dataset_train = ToxicDataset(ds_train['comment'], ds_train['toxic'], tokenizer, max_length)
dataset_test = ToxicDataset(ds_test['comment'], ds_test['toxic'], tokenizer, max_length)
dataset_val = ToxicDataset(ds_val['comment'], ds_val['toxic'], tokenizer, max_length)

### Даталоадеры

In [23]:
test_loader = DataLoader(dataset_test, batch_size)
train_loader = DataLoader(dataset_train, batch_size)
val_loader = DataLoader(dataset_val, batch_size)

### Предобученная модель DeepPavlov/rubert-base-cased

In [24]:
num_labels = len(set(ds_train['toxic']))
model = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels=num_labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Функция для подсчета метрик

In [25]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

### Кросс-валидация

#### Аргументы для кросс валидации

In [ ]:
training_args = TrainingArguments(
    output_dir="model/checkpoints",
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_dir="outputs/logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    gradient_checkpointing=True
)

#### Функция для кросс-валидации

In [27]:
def hp_space_fn(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3,log=True),
        "weight_decay" : trial.suggest_float("weight_decay", 1e-5, 0.1, log=True) 
        }

def model_init():
    return BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels=num_labels)

#### Дефолтный трейнер

In [28]:
trainer = Trainer(
    model=model,
    model_init = model_init,
    args=training_args,
    train_dataset=train_loader.dataset,  
    eval_dataset=val_loader.dataset,  
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    )

/var/folders/8d/9xq7l_zx7dn_5mb3cdq9_cgr0000gn/T/ipykernel_19364/3879862365.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/transformers/trainer.py:479: FutureWarning: `Trainer` requires either a `model` or `model_init` argument, but not both. `model_init` will overwrite your model when calling the `train` method. This will become a fatal error in the next release.
  warnings.warn(


#### Обучение с кросс-валидацией

In [ ]:
best_run = trainer.hyperparameter_search(
    hp_space=hp_space_fn,
    n_trials=5, 
    direction="maximize",
    backend="optuna" 
)

#### Результаты

In [ ]:
best_params = best_run.hyperparameters

### Обучение с оптимизатором и шедулером

In [ ]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()))
total_steps = len(dataset_train) * epochs * batch_size

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

learning_rate = 2e-5
weight_decay = 0.01

In [ ]:
training_args = TrainingArguments(
    output_dir="model/checkpoint",
    eval_strategy="epoch",
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_dir="outputs/logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    gradient_checkpointing=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
    )

### Тренировка и валидация

In [ ]:
train_metrics = trainer.train().metrics

with open("outputs/metrics/train_metrics.json", "w") as f:
    json.dump(train_metrics, f, indent=2)

eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

with open("outputs/metrics/eval_metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)

trainer.save_model("model/final_model")
tokenizer.save_pretrained("tokenizer/final_tokenizer")

### Тестирование модели

### Результаты

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_loader.dataset, metric_key_prefix="test")

with open("outputs/metrics/test_metrics.json", "w") as f:
    json.dump(test_results, f, indent=2)

{'accuracy': 0.8848821081830791,
 'f1': 0.8657798352386528,
 'precision': 0.879727720472075,
 'recall': 0.8554682742662283}